# 4. Nested Cross-Validation for Performance Evaluation

This notebook evaluates the generalization performance of the Multi-omics Integration Clustering (MIC) model using a nested cross-validation (CV) procedure. This is a robust method for estimating the performance of a model on unseen data, especially when hyperparameter tuning is involved.

### Methodology

1.  **Outer Loop (Evaluation)**: The data is split into outer folds. In each iteration, one fold is held out as the final test set, and the remaining folds are used for model training and hyperparameter selection.
2.  **Inner Loop (Hyperparameter Tuning)**: Within each outer loop iteration, the training folds are further subjected to an inner CV. This inner loop performs a grid search to find the best set of hyperparameters.
3.  **Final Evaluation**: The best hyperparameters found in the inner loop are used to train a new model on the entire training set from the outer loop. This model's final performance is then evaluated on the held-out test set.
4.  **Aggregation**: The performance metrics from the test set of each outer fold are collected and averaged to provide a stable and unbiased estimate of the model's true performance.


### 4.1. Setup and Data Loading

In [1]:
# System and project imports
import sys
import pandas as pd
import numpy as np
import torch
from tqdm.notebook import tqdm
import copy

# Append src directory to path to import custom modules
sys.path.append('../src')
import config
from models import MIC
from utils import set_seed, cluster_accuracy

# Scikit-learn imports
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score, silhouette_score

# PyTorch imports
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Subset
from torch.optim.lr_scheduler import ReduceLROnPlateau

# --- Load Processed Data ---
print("Loading preprocessed data...")
processed_data_path = config.PROCESSED_DATA_DIR / "processed_dataset.pt"
if not processed_data_path.exists():
    raise FileNotFoundError(f"Processed data not found at {processed_data_path}. Please run '01_data_preprocessing.ipynb' first.")

processed_data = torch.load(processed_data_path, map_location=torch.device('cpu'))

input_genotype = processed_data['input_genotype']
input_proteome = processed_data['input_proteome']
input_metabolite = processed_data['input_metabolite']
output_clinical = processed_data['output_clinical']
clinical_df = processed_data['clinical_df']

print("Data loaded successfully.")


Loading preprocessed data...
Data loaded successfully.


/tmp/ipykernel_246773/4144859953.py:33: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  processed_data = torch.load(processed_data_path, map_location=torch.device('cpu'))


### 4.2. Configuration for Nested CV

Define the settings for the nested cross-validation procedure, including the number of inner and outer folds, the hyperparameter search space, and fixed model parameters.


In [2]:
# --- Execution Environment ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- Nested CV Settings ---
OUTER_SPLITS = 10
INNER_SPLITS = 5


# --- Fixed Hyperparameters ---
# Note: These are based on the original study's findings
ENCODER_DIM = [128, 32]
HIDDEN_DIM = [64]
LATENT_DIM = 16
MAX_EPOCHS = 300
LEARNING_RATE = 0.01
BATCH_SIZE = 64
PATIENCE = 20

# --- Hyperparameter Search Space (for Inner Loop) ---
# This grid is kept small for demonstration purposes, but can be expanded.
param_grid = {
    'decoder_dim': [[], [32], [64]],
    'dropout': [0.1, 0.2, 0.3],
    'weight_decay': [1e-4, 1e-3, 1e-2]
}



inner_grid = list(ParameterGrid(param_grid))

print(f"\nNested CV configured: {OUTER_SPLITS} outer folds, {INNER_SPLITS} inner folds.")
print(f"Hyperparameter grid size for inner loop: {len(inner_grid)}")


Using device: cuda

Nested CV configured: 10 outer folds, 5 inner folds.
Hyperparameter grid size for inner loop: 27


### 4.3. Training Function for a Single Fold

This is the core function that trains and evaluates the model on a given training and validation/test set. It is adapted from the original `hyperparameter_tuning.ipynb` notebook to ensure logical consistency. It returns a detailed tuple of metrics for thorough logging.


In [3]:
def train_one_fold(train_idx, val_idx, params):
    """
    Trains and evaluates the MIC model for one fold, adapted from the original notebook's logic.
    """
    # --- Model Definition ---
    input_dims = {
        'genotype': input_genotype.shape[1],
        'proteome': input_proteome.shape[1],
        'metabolite': input_metabolite.shape[1]
    }

    model = MIC(
        input_dims=input_dims,
        encoder_dims=config.ENCODER_DIMS,
        integration_dims=config.INTEGRATION_DIMS,
        latent_dim=config.LATENT_DIM,
        decoder_dims=params['decoder_dim'], # Tuned hyperparameter
        clinical_output_dim=config.CLINICAL_OUTPUT_DIM,
        cluster_num=config.NUM_CLUSTERS,
        dropout=params['dropout'] # Tuned hyperparameter
    ).to(DEVICE)
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=params['weight_decay'])
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.9, patience=5)
    loss_fn = F.mse_loss

    # --- DataLoaders ---
    full_dataset = TensorDataset(input_genotype, input_proteome, input_metabolite, output_clinical)
    train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader   = DataLoader(Subset(full_dataset, val_idx),   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    eval_loader  = DataLoader(Subset(full_dataset, train_idx),  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

    # --- Training Loop with Early Stopping ---
    best_val_loss = float('inf')
    epochs_no_improve = 0
    best_model_wts = model.state_dict()
    trained_epochs = 0
    
    for epoch in range(MAX_EPOCHS):
        model.train()
        for x1, x2, x3, y in train_loader:
            x1, x2, x3, y = x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            pred = model(x1, x2, x3)
            loss = loss_fn(pred, y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        epoch_val_losses = []
        with torch.no_grad():
            for x1, x2, x3, y in val_loader:
                pred = model(x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE))
                epoch_val_losses.append(loss_fn(pred, y.to(DEVICE)).item())
        
        current_val_loss = np.mean(epoch_val_losses)
        scheduler.step(current_val_loss)
        
        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            epochs_no_improve = 0
            best_model_wts = model.state_dict()
        else:
            epochs_no_improve += 1
        
        if epochs_no_improve >= PATIENCE:
            trained_epochs = epoch + 1
            break
            
    if trained_epochs == 0:
        trained_epochs = MAX_EPOCHS
            
    model.load_state_dict(best_model_wts)
    model.eval()

    # --- Final Loss Calculation ---
    final_train_loss_list = []
    with torch.no_grad():
        for x1, x2, x3, y in train_loader:
            pred = model(x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE))
            final_train_loss_list.append(loss_fn(pred, y.to(DEVICE)).item())
    final_train_loss = np.mean(final_train_loss_list)

    final_val_loss_list = []
    with torch.no_grad():
        for x1, x2, x3, y in val_loader:
            pred = model(x1.to(DEVICE), x2.to(DEVICE), x3.to(DEVICE))
            final_val_loss_list.append(loss_fn(pred, y.to(DEVICE)).item())
    final_val_loss = np.mean(final_val_loss_list)

    # --- Clustering Metrics Calculation ---
    train_cluster_acc, val_cluster_acc, train_ari, val_ari = -1, -1, -1, -1
    train_sil_score, val_sil_score, train_nmi, val_nmi, train_ami, val_ami = -1, -1, -1, -1, -1, -1
    
    with torch.no_grad():
        z_train = model.get_latent_space(eval_loader, device=DEVICE)
        kmeans_model = KMeans(n_clusters=config.NUM_CLUSTERS, random_state=config.RANDOM_STATE, n_init=100).fit(z_train.numpy())
        train_labels_pred = kmeans_model.predict(z_train.numpy())
        train_labels_true = clinical_df.iloc[train_idx]["kmeans_cluster"].values
        
        train_cluster_acc = cluster_accuracy(train_labels_true, train_labels_pred)
        train_ari = adjusted_rand_score(train_labels_true, train_labels_pred)
        if len(np.unique(train_labels_pred)) > 1:
            train_sil_score = silhouette_score(z_train.numpy(), train_labels_pred)
        train_nmi = normalized_mutual_info_score(train_labels_true, train_labels_pred)
        train_ami = adjusted_mutual_info_score(train_labels_true, train_labels_pred)
        
        z_val = model.get_latent_space(val_loader, device=DEVICE)
        val_labels_pred = kmeans_model.predict(z_val.numpy())
        val_labels_true = clinical_df.iloc[val_idx]["kmeans_cluster"].values
        
        val_cluster_acc = cluster_accuracy(val_labels_true, val_labels_pred)
        val_ari = adjusted_rand_score(val_labels_true, val_labels_pred)
        if len(np.unique(val_labels_pred)) > 1:
            val_sil_score = silhouette_score(z_val.numpy(), val_labels_pred)
        val_nmi = normalized_mutual_info_score(val_labels_true, val_labels_pred)
        val_ami = adjusted_mutual_info_score(val_labels_true, val_labels_pred)

    return (final_train_loss, final_val_loss, 
            train_cluster_acc, val_cluster_acc, 
            train_ari, val_ari, 
            train_sil_score, val_sil_score,
            train_nmi, val_nmi,
            train_ami, val_ami,
            trained_epochs,
            val_labels_true,
            val_labels_pred)


### 4.4. Run Nested Cross-Validation Loop

This is the main execution block. It iterates through the outer folds, and for each one, it runs a complete inner cross-validation loop to find the best hyperparameters. Then, it trains the model with these optimal parameters on the full training data of the outer fold and evaluates it on the held-out test set.

This logic is adapted from the original `hyperparameter_tuning.ipynb` to ensure consistency.


In [4]:
set_seed(config.RANDOM_STATE)
outer_kfold = StratifiedKFold(n_splits=OUTER_SPLITS, shuffle=True, random_state=42)
y_all = clinical_df["kmeans_cluster"].values

outer_results = []

print(f"Starting Nested CV: Outer {OUTER_SPLITS}-Fold / Inner {INNER_SPLITS}-Fold")
print("="*90)

for outer_fold, (train_val_idx, test_idx) in enumerate(tqdm(outer_kfold.split(np.zeros(len(y_all)), y_all), total=OUTER_SPLITS, desc="Outer Folds"), 1):
    
    best_inner_score = -np.inf
    best_inner_params = None
    
    # --- Inner Loop for Hyperparameter Search ---
    inner_kfold = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=42)
    y_outer_train = y_all[train_val_idx]
    
    for params in inner_grid:
        inner_scores = []
        for inner_train_idx, inner_val_idx in inner_kfold.split(np.zeros(len(y_outer_train)), y_outer_train):
            train_idx = train_val_idx[inner_train_idx]
            val_idx   = train_val_idx[inner_val_idx]
            
            # Unpack the large tuple, we only need val_acc for inner loop scoring
            _, _, _, val_acc, _, _, _, _, _, _, _, _, _, _, _ = train_one_fold(train_idx, val_idx, params)
            inner_scores.append(val_acc)
        
        mean_inner_acc = np.mean(inner_scores)
        if mean_inner_acc > best_inner_score:
            best_inner_score = mean_inner_acc
            best_inner_params = params
            
    # --- Outer Fold Evaluation ---
    # Train on the full train_val set with the best params and evaluate on the test set
    _, _, _, test_acc, _, test_ari, _, test_sil, _, test_nmi, _, test_ami, _, _, _ = train_one_fold(train_val_idx, test_idx, best_inner_params)
    
    result_entry = {
        'fold': outer_fold,
        'best_params': str(best_inner_params),
        'test_acc': test_acc,
        'test_ari': test_ari,
        'test_nmi': test_nmi,
        'test_ami': test_ami,
        'test_silhouette': test_sil
    }
    outer_results.append(result_entry)
    
    tqdm.write(f"[Outer Fold {outer_fold}/{OUTER_SPLITS}] Best Inner Params: {best_inner_params} (Validation ACC: {best_inner_score:.4f}) | Test ACC: {test_acc:.4f}")

# --- Final Summary ---
df_outer = pd.DataFrame(outer_results)
print("\n" + "="*90)
print("===== Nested CV Summary (Outer Fold Test Performance) =====")
display(df_outer)

print("\n--- Average Performance Metrics across Outer Folds ---")
print("Averages:")
print(df_outer.mean(numeric_only=True).round(4).to_string())
print("\nStandard Deviations:")
print(df_outer.std(numeric_only=True).round(4).to_string())
print("="*90)

# Save results
results_path = config.OUTPUT_DIR / "nested_cv_results.csv"
# df_outer.to_csv(results_path, index=False)
print(f"\nNested CV results saved to:\n{results_path}")


Starting Nested CV: Outer 10-Fold / Inner 5-Fold


Outer Folds:   0%|          | 0/10 [00:00<?, ?it/s]

[Outer Fold 1/10] Best Inner Params: {'decoder_dim': [], 'dropout': 0.3, 'weight_decay': 0.0001} (Validation ACC: 0.5733) | Test ACC: 0.4000
[Outer Fold 2/10] Best Inner Params: {'decoder_dim': [], 'dropout': 0.3, 'weight_decay': 0.0001} (Validation ACC: 0.5467) | Test ACC: 0.4800
[Outer Fold 3/10] Best Inner Params: {'decoder_dim': [], 'dropout': 0.2, 'weight_decay': 0.01} (Validation ACC: 0.5333) | Test ACC: 0.6800
[Outer Fold 4/10] Best Inner Params: {'decoder_dim': [], 'dropout': 0.1, 'weight_decay': 0.001} (Validation ACC: 0.5511) | Test ACC: 0.5000
[Outer Fold 5/10] Best Inner Params: {'decoder_dim': [], 'dropout': 0.3, 'weight_decay': 0.001} (Validation ACC: 0.5400) | Test ACC: 0.5400
[Outer Fold 6/10] Best Inner Params: {'decoder_dim': [], 'dropout': 0.1, 'weight_decay': 0.0001} (Validation ACC: 0.5533) | Test ACC: 0.4800
[Outer Fold 7/10] Best Inner Params: {'decoder_dim': [], 'dropout': 0.1, 'weight_decay': 0.0001} (Validation ACC: 0.5600) | Test ACC: 0.5000
[Outer Fold 8/10]

,fold,best_params,test_acc,test_ari,test_nmi,test_ami,test_silhouette
0,1,"{'decoder_dim': [], 'dropout': 0.3, 'weight_de...",0.40,0.007409,0.093312,0.009268,0.120543
1,2,"{'decoder_dim': [], 'dropout': 0.3, 'weight_de...",0.48,0.043655,0.108116,0.047926,0.188069
2,3,"{'decoder_dim': [], 'dropout': 0.2, 'weight_de...",0.68,0.288124,0.313568,0.266178,0.148508
3,4,"{'decoder_dim': [], 'dropout': 0.1, 'weight_de...",0.50,0.102002,0.082161,0.021963,0.182948
4,5,"{'decoder_dim': [], 'dropout': 0.3, 'weight_de...",0.54,0.118870,0.159181,0.085505,0.241229
5,6,"{'decoder_dim': [], 'dropout': 0.1, 'weight_de...",0.48,0.028959,0.069156,0.006097,0.144987
6,7,"{'decoder_dim': [], 'dropout': 0.1, 'weight_de...",0.50,0.044940,0.095527,0.018420,0.018814
7,8,"{'decoder_dim': [], 'dropout': 0.1, 'weight_de...",0.62,0.197727,0.213108,0.152186,0.248850
8,9,"{'decoder_dim': [32], 'dropout': 0.3, 'weight_...",0.56,0.237447,0.356090,0.296199,0.246599
9,10,"{'decoder_dim': [], 'dropout': 0.2, 'weight_de...",0.46,0.070166,0.084573,0.024530,0.226856



--- Average Performance Metrics across Outer Folds ---
Averages:
fold               5.5000
test_acc           0.5220
test_ari           0.1139
test_nmi           0.1575
test_ami           0.0928
test_silhouette    0.1767

Standard Deviations:
fold               3.0277
test_acc           0.0813
test_ari           0.0960
test_nmi           0.1033
test_ami           0.1089
test_silhouette    0.0721

Nested CV results saved to:
/data02/jaejoon/T2D_subtype_analysis/outputs/nested_cv_results.csv


### 4.5. Performance Comparison with Benchmarks

To contextualize the performance of our multi-omics integration model, we compare it against two benchmark methods:
1.  **Random Chance**: A baseline model that assigns subtype labels randomly, while preserving the original subtype distribution. This helps establish the lower bound for performance.
2.  **PCA + K-means**: A standard dimensionality reduction approach where we first apply Principal Component Analysis (PCA) to the concatenated multi-omics data and then perform K-means clustering on the principal components.


In [2]:
print("Starting benchmark analysis...")

# Extra imports for benchmark analysis
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.utils import check_random_state
from collections import Counter


# Load the results from the nested CV of our model
nested_cv_results = pd.read_csv(f"{config.OUTPUT_DIR}/nested_cv_results.csv")

if nested_cv_results is not None:
    # Define true labels from the clinical dataframe
    y_true = clinical_df["kmeans_cluster"]

    # --- 1. Random Chance Benchmark ---
    print("\nRunning Random Chance benchmark...")

    def stratified_random_labels(y_true, rng):
        """Generates random labels while preserving class distribution."""
        classes, counts = np.unique(y_true, return_counts=True)
        probs = counts / counts.sum()
        return rng.choice(classes, size=len(y_true), p=probs)

    B = 10  # Number of random iterations
    rng = check_random_state(config.RANDOM_STATE)

    random_metrics = {
        'acc': [], 'ari': [], 'nmi': [], 'ami': []
    }

    for _ in range(B):
        y_rand = stratified_random_labels(y_true, rng)
        random_metrics['acc'].append(cluster_accuracy(y_true, y_rand))
        random_metrics['ari'].append(adjusted_rand_score(y_true, y_rand))
        random_metrics['nmi'].append(normalized_mutual_info_score(y_true, y_rand))
        random_metrics['ami'].append(adjusted_mutual_info_score(y_true, y_rand))

    random_df = pd.DataFrame(random_metrics)
    print("Random Chance benchmark complete.")

    # --- 2. PCA + K-means Benchmark ---
    print("\nRunning PCA + K-means benchmark...")
    
    # Combine all omics data
    X_combined = torch.cat([
        processed_data['input_genotype'], 
        processed_data['input_proteome'], 
        processed_data['input_metabolite']
    ], dim=1)
    X_np = X_combined.numpy()

    # Apply PCA to retain 95% of variance
    pca = PCA(n_components=0.95, random_state=config.RANDOM_STATE)
    X_pca = pca.fit_transform(X_np)
    print(f"PCA reduced dimensions from {X_np.shape[1]} to {X_pca.shape[1]}")

    pca_metrics = {
        'acc': [], 'ari': [], 'nmi': [], 'ami': []
    }

    # Run K-means 10 times with different seeds for stability
    for i in range(10):
        kmeans = KMeans(n_clusters=config.NUM_CLUSTERS, n_init=100)
        cluster_labels = kmeans.fit_predict(X_pca)
        
        pca_metrics['acc'].append(cluster_accuracy(y_true, cluster_labels))
        pca_metrics['ari'].append(adjusted_rand_score(y_true, cluster_labels))
        pca_metrics['nmi'].append(normalized_mutual_info_score(y_true, cluster_labels))
        pca_metrics['ami'].append(adjusted_mutual_info_score(y_true, cluster_labels))

    pca_df = pd.DataFrame(pca_metrics)
    print("PCA + K-means benchmark complete.")


Starting benchmark analysis...

Running Random Chance benchmark...
Random Chance benchmark complete.

Running PCA + K-means benchmark...
PCA reduced dimensions from 1423 to 367
PCA + K-means benchmark complete.


### 4.6. Statistical Analysis and Result Summary

After calculating the performance for all three methods (Multi-omics, PCA+K-means, Random), we perform a statistical test to validate if the performance improvement of our model is significant.

We use an independent t-test to compare the accuracy of the Multi-omics model against the benchmark models. To account for multiple comparisons (Multi-omics vs. PCA and Multi-omics vs. Random), we apply the Benjamini/Hochberg FDR correction to the p-values.

Finally, we compile a summary table of all metrics (mean and standard deviation) and save both the statistical results and the summary table to the `outputs` directory.


In [3]:
# Extra imports for statistical analysis
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

if nested_cv_results is not None:
    print("Starting statistical analysis...")
    
    # --- 1. Compile data for comparison ---
    metrics = ["acc", "ari", "nmi", "ami"]
    raw_data_dict = {}

    for metric in metrics:
        raw_data_dict[metric] = {
            'Multi-omics': nested_cv_results[f"test_{metric}"].values,
            'PCA+Kmeans': pca_df[metric].values,
            'Random': random_df[metric].values
        }

    # --- 2. Perform t-tests on Accuracy ---
    comparisons = [
        ('Multi-omics', 'PCA+Kmeans'),
        ('Multi-omics', 'Random')
    ]

    stat_results = []
    p_values = []

    for method1, method2 in comparisons:
        # We use a one-sided t-test ("greater") to test if Multi-omics accuracy is greater than the benchmark's.
        stat, p_value = ttest_ind(
            raw_data_dict["acc"][method1], 
            raw_data_dict["acc"][method2],
            alternative="greater",
            equal_var=False  # Welch's t-test, as sample sizes/variances might differ
        )
        p_values.append(p_value)
        stat_results.append({
            'Comparison': f'{method1} vs {method2}',
            'Statistic': stat,
            'p-value': p_value
        })

    # Apply FDR correction
    _, p_values_corrected, _, _ = multipletests(p_values, method='fdr_bh')

    for i, result in enumerate(stat_results):
        result['FDR Corrected p-value'] = p_values_corrected[i]

    stats_df = pd.DataFrame(stat_results)
    print("\n--- T-test Results (Accuracy) ---")
    print(stats_df)

    # --- 3. Print Significance Summary ---
    def get_significance_marker(p_value):
        if p_value < 0.001: return '***'
        if p_value < 0.01: return '**'
        if p_value < 0.05: return '*'
        return 'ns' # not significant

    print("\n--- Significance of Multi-omics Model ---")
    for i, (method1, method2) in enumerate(comparisons):
        significance = get_significance_marker(p_values_corrected[i])
        print(f"{method1} vs {method2}: {significance} (FDR p-value: {p_values_corrected[i]:.4f})")

    # --- 4. Create and Save Final Summary Table ---
    summary_df = pd.DataFrame()
    methods = ['Multi-omics', 'PCA+Kmeans', 'Random']
    
    for metric in metrics:
        for method in methods:
            summary_df.loc[method, f"{metric}_mean"] = np.mean(raw_data_dict[metric][method])
            summary_df.loc[method, f"{metric}_std"] = np.std(raw_data_dict[metric][method])
    
    # Save the results
    summary_path = f"{config.OUTPUT_DIR}/benchmark_comparison_metrics.csv"
    stats_path = f"{config.OUTPUT_DIR}/benchmark_comparison_stats.csv"
    
    summary_df.to_csv(summary_path, index=True)
    stats_df.to_csv(stats_path, index=False)

    print(f"\nSuccessfully saved benchmark metrics to: {summary_path}")
    print(f"Successfully saved statistical results to: {stats_path}")
else:
    print("Skipping statistical analysis as Nested CV results are not available.")


Starting statistical analysis...

--- T-test Results (Accuracy) ---
                  Comparison  Statistic       p-value  FDR Corrected p-value
0  Multi-omics vs PCA+Kmeans  16.909989  5.598641e-09           5.598641e-09
1      Multi-omics vs Random  15.127691  6.218890e-11           1.243778e-10

--- Significance of Multi-omics Model ---
Multi-omics vs PCA+Kmeans: *** (FDR p-value: 0.0000)
Multi-omics vs Random: *** (FDR p-value: 0.0000)

Successfully saved benchmark metrics to: /data02/jaejoon/T2D_subtype_analysis/outputs/benchmark_comparison_metrics.csv
Successfully saved statistical results to: /data02/jaejoon/T2D_subtype_analysis/outputs/benchmark_comparison_stats.csv
